# 4. Evaluate

This notebook benchmarks serving endpoint latency and checks vector search retrieval quality.

In [ ]:
%pip install uv
%sh uv pip install .
%sh uv pip install ".[local]"
%restart_python

In [ ]:
import statistics
import time

import os
from pathlib import Path
from mlflow.deployments import get_deploy_client

from utils import is_local_env, resolve_output_root, resolve_sample_inputs

config = mlflow.models.ModelConfig(development_config="./config.yaml")
config = config.to_dict()

is_local = is_local_env()

if is_local:
    from databricks.connect import DatabricksSession

    spark = DatabricksSession.builder.getOrCreate()

ENDPOINT_NAME = config["serving_endpoint"]

local_input_path = config.get("local_input_path")
local_output_path = config.get("local_output_path")

OUTPUT_ROOT = str(resolve_output_root(
    config["catalog"],
    config["schema"],
    config["output_volume"],
    local_output_path,
))

if is_local and local_input_path:
    default_sample_inputs = [str(Path(local_input_path).expanduser().resolve() / "sample.pdf")]
else:
    default_sample_inputs = [
        f"/Volumes/{config['catalog']}/{config['schema']}/{config['input_volume']}/sample.pdf"
    ]

sample_inputs = config.get("evaluate_sample_inputs", default_sample_inputs)
sample_inputs = resolve_sample_inputs(sample_inputs, local_input_path)

rows = [[path, OUTPUT_ROOT, {}] for path in sample_inputs]
request = {
    "dataframe_split": {
        "columns": ["file_path", "output_root", "options"],
        "data": rows,
    }
}

deploy_client = get_deploy_client("databricks")

start = time.perf_counter()
response = deploy_client.predict(endpoint=ENDPOINT_NAME, inputs=request)
elapsed = time.perf_counter() - start

print(f"Calls: {len(sample_inputs)}")
print(f"Mean: {elapsed:.3f}s")

In [ ]:
from databricks.vector_search.client import VectorSearchClient

VECTOR_SEARCH_ENDPOINT = config["vector_search_endpoint"]
INDEX_NAME = config["index_name"]
CATALOG = config["catalog"]
SCHEMA = config["schema"]

index_fullname = f"{CATALOG}.{SCHEMA}.{INDEX_NAME}"

vsc = VectorSearchClient()
index = vsc.get_index(endpoint_name=VECTOR_SEARCH_ENDPOINT, index_name=index_fullname)

query_text = config.get("evaluate_query_text", "maintenance schedule")
results = index.similarity_search(query_text=query_text, columns=["doc_path", "content"], num_results=5)

print(results)

In [ ]:
rows = results.get("result", results)

if isinstance(rows, dict) and "data_array" in rows:
    records = rows["data_array"]
    columns = rows.get("columns", [])
else:
    records = rows
    columns = []

print("Top matches:")
for record in records[:5]:
    if columns:
        entry = dict(zip(columns, record))
        print(entry.get("doc_path"))
    else:
        print(record)

In [ ]:
from pyspark.sql.functions import col, lower, regexp_extract, regexp_replace, trim

CATALOG = config["catalog"]
SCHEMA = config["schema"]
INPUT_VOLUME = config["input_volume"]
PROCESSED_TABLE = config["parsed_markdown_table"]
AIPARSE_TABLE = config["aiparse_outputs_table"]

raw_glob = f"/Volumes/{CATALOG}/{SCHEMA}/{INPUT_VOLUME}/*"

parsed_df = spark.sql(
    f"""
    SELECT
      path,
      ai_parse_document(content, map('version', '2.0')) AS parsed
    FROM read_files('{raw_glob}', format => 'binaryFile')
    """
)

def normalize_stem(path_col):
    stem = regexp_extract(path_col, r"([^/]+)$", 1)
    stem = regexp_replace(stem, r"\.\w+$", "")
    stem = regexp_replace(stem, "%20", "_")
    stem = regexp_replace(stem, r"[^\\w\\s-]", "")
    stem = regexp_replace(stem, r"\\s+", "_")
    stem = regexp_replace(stem, r"_+", "_")
    stem = trim(regexp_replace(stem, r"^_+|_+$", ""))
    return lower(stem)

parsed_df = parsed_df.withColumn("doc_key", normalize_stem(col("path")))
parsed_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.{AIPARSE_TABLE}")

md_df = spark.table(f"{CATALOG}.{SCHEMA}.{PROCESSED_TABLE}")
md_df = md_df.withColumn("doc_key", normalize_stem(col("doc_path")))

comparison_df = (
    parsed_df.join(md_df, on="doc_key", how="inner")
    .select(
        col("path").alias("raw_path"),
        col("doc_path"),
        col("parsed").alias("aiparse_parsed"),
        col("content").alias("docling_md"),
    )
)

comparison_df.createOrReplaceTempView("doc_compare")
print("Comparison rows:", comparison_df.count())

In [ ]:
import json

from mlflow.deployments import get_deploy_client
from pyspark.sql.functions import to_json

LLM_ENDPOINT = config["llm_endpoint"]
MAX_COMPARE = int(config.get("evaluate_max_compare", 3))

sample_rows = (
    comparison_df
    .withColumn("aiparse_json", to_json(col("aiparse_parsed")))
    .select("raw_path", "docling_md", "aiparse_json")
    .limit(MAX_COMPARE)
    .collect()
)

deploy_client = get_deploy_client("databricks")
comparisons = []

for row in sample_rows:
    prompt = f"""
Compare the two parsing outputs from the same document.

Return JSON with these keys:
- overall_preference: one of ["ai_parse_document", "docling", "tie"]
- ai_parse_score: integer 1-5
- docling_score: integer 1-5
- differences: short list of key differences
- notes: short summary

ai_parse_document JSON:
{row.aiparse_json[:6000]}

Docling markdown:
{row.docling_md[:6000]}
"""

    response = deploy_client.predict(
        endpoint=LLM_ENDPOINT,
        inputs={"messages": [{"role": "user", "content": prompt}]},
    )

    if isinstance(response, dict) and "choices" in response:
        content = response["choices"][0]["message"]["content"]
    elif isinstance(response, dict) and "output" in response:
        content = response["output"]
    else:
        content = str(response)

    comparisons.append({"raw_path": row.raw_path, "llm_comparison": content})

print(json.dumps(comparisons, indent=2))